# 4 · SHAP: fiyatı ne belirliyor

SHAP raporunun kaynakları ([shap.tr.md](../reports/shap.tr.md)).

**Ne koşar, ne kayıttan gelir:**
- **Hafif betikler bu defterde koşar.** `script = run("<betik>")` betiğin [1]–[5] hücrelerini çalıştırır: veriyi ve
  kayıtlı model çıktılarını (OOF tahminleri, SHAP dosyası) okur, hesabı yapar; kaydet hücresi [6] çalışmaz.
  Ardından `check` sonucun `metrics/` kaydıyla aynı olup olmadığını gösterir. `script.res`, yüklenen tablolar
  (`script.listings` …) ve betiğin bütün fonksiyonları sonraki hücrelerde kullanılabilir.
- **Ağır betikler kayıttan okunur.** Model eğiten ya da çapraz doğrulama koşan betikler (`HEAVY`) için
  `recorded("<betik>", docs)` `metrics/` kaydını getirir.
- **Soru, veri, yöntem ve sızıntı notları** kartlardan (`analysis/cards/`), **figürler** derleyicilerin çizdiği
  `reports/figures/`'tan gelir.

Analiz kodu betiklerin kendisidir; ortak kod `analysis/lib` ve `builders/report_lib`'den gelir, burada kopyası yok. Bir
sonucu kalıcı olarak değiştirmek için betik değişir ve zincir koşulur; defter yeni kaydı okur. Defter dosya yazmaz.

In [1]:
import sys
import warnings
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis" / "lib").is_dir())
sys.path[:0] = [str(ROOT / "analysis"), str(ROOT / "builders")]
# shap kütüphanesi import edilirken ipywidgets yok uyarısı verir; çıktıyı etkilemez
warnings.filterwarnings("ignore", message="IProgress not found")

import pandas as pd
from IPython.display import Markdown

from lib.labels import lb
from lib.records import card, check, flat, recorded, run, runs, show
from report_lib import metrics_view as MV

pd.set_option("display.max_rows", 30)
pd.set_option("display.max_colwidth", 120)
SCRIPTS = ["shap/02_oof_shap", "shap/03_what_sets_price", "shap/04_variants", "shap/06_one_listing"]
docs = MV.load_metrics(ROOT / "metrics")
MV.check_consistency(docs)
runs(docs, SCRIPTS)

,yazıldı,veri sonu,DB,run_id
betik,,,,
shap/02_oof_shap,2026-09-27T21:59:31+03:00,2026-06-27,444123b1f1a9,601741517e09
shap/03_what_sets_price,2026-11-22T18:55:49+03:00,2026-06-27,444123b1f1a9,601741517e09
shap/04_variants,2026-09-29T00:05:25+03:00,2026-06-27,444123b1f1a9,NaN
shap/06_one_listing,2026-11-22T18:55:52+03:00,2026-06-27,444123b1f1a9,601741517e09


### shap/02_oof_shap
[betik](../analysis/shap/02_oof_shap.py) · [kart](../analysis/cards/shap/02_oof_shap.json)

In [2]:
Markdown(card("shap/02_oof_shap"))

**Soru:** Model tahminlerini hangi öznitelikler sürüklüyor, her ilanı onu görmemiş fold modeliyle açıklayınca?

**Veri:** load_clean() (29.988 ilan) ve 07_model_comparison'ın OOF artefaktı (data/analysis/oof.parquet, run_id kapısı).

**Yöntem:** Beş fold modeli aynı parametrelerle yeniden kurulur; saklı OOF tahminlerini eşikler içinde üretmezlerse durur (kapı). Her ilanın SHAP değerleri kendi fold modelinden; grup SHAP = satır başına üye SHAP toplamı; genel önem = ortalama |SHAP|, gruplu ve ayrı. Ham SHAP diziler data/analysis/oof_shap.npz'ye yazılır.

**Bölme:** `kfold_oof`

**Sızıntı:**
- aynı ilan iki tarafta: `present` — Farklı ad_id ile yeniden ilan edilen aynı araç fold'lar arasına düşebilir: 01_dedup_leakage katı tanımda %0,46, fiyatsız tanımda %4,26 ölçüyor; etkisi ölçülmedi.
- hedef öznitelikte: `none` — Öznitelik listesinde (lib/common.FEATURES) fiyat ya da fiyattan türeyen bir alan yok.
- bütün veriye uydurma: `present` — Kategori kodları, TF-IDF/SVD ve tabanlar yalnız eğitim fold'unda kurulur (lib/cv.fold_matrices). Ama LightGBM'in erken durdurması her fold'un kendi doğrulama kısmında (lib/cv.fit_lgb_fold): ağaç sayısı skorlanan veriyle seçiliyor, OOF hatası hafifçe iyimser olabilir. Etkisi ölçülmedi (iç içe bölmeyle ölçülebilir).
- zamansal: `present` — KFold dönemleri karıştırır; model zamanı görmez. Medyan artık dönemlere göre kayıyor (08_residuals); ileriye dönük hata 09_backtest'te ayrıca ölçülüyor.

In [3]:
m = recorded("shap/02_oof_shap", docs)

**Kayıttan okunuyor** (ağır betik: fold modellerini yeniden kurar ve SHAP hesaplar; oof_shap.npz'yi yazar).

In [4]:
o = m["oof_shap"]
cols = ["grup", "kimlik", "ortalama |SHAP|", "pay %"]
show("özet ve kapı", flat(o))
show("küresel önem (grup içi işaretli toplam)", pd.DataFrame([[lb(g, "tr"), g, v, p] for g, v, p in o["global"]], columns=cols))
show("küresel önem (her üye ayrı)", pd.DataFrame([[lb(g, "tr"), g, v, p] for g, v, p in o["global_separate"]], columns=cols))

**özet ve kapı**

,değer
gate.max_gap_tl,0.0
gate.mean_gap_tl,0.0
gate.mape_refit,6.4898
gate.mape_published,6.49
gate.n_threads,16
gate.threshold_max_tl,1.0
gate.threshold_mape,0.01
n,29988
grouping_rule,"kuresel = mean|sum phi| (grup ici isaretli toplam, sonra mutlak ortalama); kuresel_ayri = sum mean|phi| (her uye ayr..."
note,"Her ilan, onu egitimde hic gormemis fold modeliyle aciklandi (5-fold, ureticinin kurulumunun aynisi). SHAP degerleri..."


**küresel önem (grup içi işaretli toplam)**

,grup,kimlik,ortalama |SHAP|,pay %
0,Yaş (yıl),vehicle_age,0.2870,42.0
1,Motor gücü (hp),power_hp_val,0.1324,19.4
2,Kilometre,gb_mileage,0.0842,12.3
3,Model/seri adı,MODEL_SERIES (text),0.0666,9.8
4,Segment,segment,0.0282,4.1
5,Kaput,hood_state,0.0169,2.5
6,Ağır hasar,is_heavy_damaged,0.0103,1.5
7,Kapı boyalı,door_painted,0.0088,1.3
8,Çamurluk değişen,fender_changed,0.0084,1.2
9,Çamurluk boyalı,fender_painted,0.0082,1.2


**küresel önem (her üye ayrı)**

,grup,kimlik,ortalama |SHAP|,pay %
0,Yaş (yıl),vehicle_age,0.2870,36.7
1,Model/seri adı,MODEL_SERIES (text),0.1659,21.2
2,Motor gücü (hp),power_hp_val,0.1324,16.9
3,Kilometre,gb_mileage,0.0842,10.8
4,Segment,segment,0.0282,3.6
5,Kaput,hood_state,0.0169,2.2
6,Ağır hasar,is_heavy_damaged,0.0103,1.3
7,Kapı boyalı,door_painted,0.0088,1.1
8,Çamurluk değişen,fender_changed,0.0084,1.1
9,Çamurluk boyalı,fender_painted,0.0082,1.1


### shap/03_what_sets_price
[betik](../analysis/shap/03_what_sets_price.py) · [kart](../analysis/cards/shap/03_what_sets_price.json)

In [5]:
Markdown(card("shap/03_what_sets_price"))

**Soru:** OOF SHAP'a göre fiyatı ne belirliyor: sıralama, yön, tipik etki, yaş/güç/km bağımlılığı ve fold'lar arası kararlılık?

**Veri:** load_clean() ve 02_oof_shap'ın SHAP dizileri (data/analysis/oof_shap.npz, run_id kapısı).

**Yöntem:** Grup önem tablosu, fold başına sıralama, yön (Spearman), tipik etki (fiyattaki karşılık), yaş bantları, güç bantları, km pencereleri, yaş × km, iki dilli figürler.

**Bölme:** `oof_input`

**Sızıntı:**
- aynı ilan iki tarafta: `n/a`
- hedef öznitelikte: `none` — SHAP, fiyattan türeyen öznitelik içermeyen modelin tahminlerini açıklıyor.
- bütün veriye uydurma: `n/a`
- zamansal: `n/a`

In [6]:
script = run("shap/03_what_sets_price")           # [1]–[5]: yükle + hesapla; kaydetmez
m = script.to_metrics(script.res)
check(m, docs["shap/03_what_sets_price"])

top | en büyük: [['vehicle_age', 0.287, 43.9], ['ENGINE', 0.1347, 20.6], ['gb_mileage', 0.0842, 12.9]] · directions | yönler: {'vehicle_age': -0.99, 'gb_mileage': -0.96, 'power_hp_val': 0.89}


**Kayıtla aynı:** defterde hesaplanan sonuç `metrics/` kaydıyla birebir aynı.

In [7]:
s = m["shap"]
cols = ["grup", "kimlik", "ortalama |SHAP|", "pay %"]
show("kaba gruplar", pd.DataFrame([[lb(g, "tr"), g, v, p] for g, v, p in s["lightgbm_tfidf_svd"]], columns=cols))
show("kaba gruplar (her üye ayrı)", pd.DataFrame([[lb(g, "tr"), g, v, p] for g, v, p in s["oof_separate"]], columns=cols))
show("fold başına sıra", pd.DataFrame(s["fold_ranks"], index=[f"fold {i}" for i in range(1, 6)],
                                     columns=range(1, len(s["fold_ranks"][0]) + 1)))
show("tipik fiyat etkisi", pd.DataFrame(s["typical"], index=["medyan |e^s − 1| %", "aşağı itilen pay %"]).T)
show("yön, bağımlılık, km × yaş", flat({"direction": s["direction"], "dep_facts": s["dep_facts"], "km_age": s["km_age"],
                                        "cohorts": s["cohorts"]["tr"], "n": s["n"], "n_svd": s["n_svd"]}))
show("motor gücü bantları", pd.DataFrame(s["dep_facts"]["hp_bands"], columns=["alt hp", "üst hp", "ortalama SHAP"]))
show("yaş bantları", pd.DataFrame(s["dep_facts"]["age_abs"], columns=["alt yaş", "üst yaş", "ortalama SHAP"]))

**kaba gruplar**

,grup,kimlik,ortalama |SHAP|,pay %
0,Yaş (yıl),vehicle_age,0.2870,43.9
1,Motor (hp + cc),ENGINE,0.1347,20.6
2,Kilometre,gb_mileage,0.0842,12.9
3,Model/seri adı,MODEL_SERIES (text),0.0666,10.2
4,Hasar (panel + ağır hasar),DAMAGE,0.0412,6.3
5,Segment,segment,0.0282,4.3
6,Kasa tipi,kb_body_type,0.0070,1.1
7,Vites,kb_transmission,0.0043,0.7
8,Yakıt,kb_fuel,0.0009,0.1
9,Çekiş,kb_drivetrain,0.0001,0.0


**kaba gruplar (her üye ayrı)**

,grup,kimlik,ortalama |SHAP|,pay %
0,Yaş (yıl),vehicle_age,0.2870,36.7
1,Model/seri adı,MODEL_SERIES (text),0.1659,21.2
2,Motor (hp + cc),ENGINE,0.1376,17.6
3,Kilometre,gb_mileage,0.0842,10.8
4,Hasar (panel + ağır hasar),DAMAGE,0.0672,8.6
5,Segment,segment,0.0282,3.6
6,Kasa tipi,kb_body_type,0.0070,0.9
7,Vites,kb_transmission,0.0043,0.5
8,Yakıt,kb_fuel,0.0009,0.1
9,Çekiş,kb_drivetrain,0.0001,0.0


**fold başına sıra**

,1,2,3,4,5,6,7,8,9,10,11
fold 1,vehicle_age,ENGINE,MODEL_SERIES (text),gb_mileage,DAMAGE,segment,kb_body_type,kb_transmission,kb_fuel,kb_drivetrain,brand
fold 2,vehicle_age,ENGINE,gb_mileage,MODEL_SERIES (text),DAMAGE,segment,kb_body_type,kb_transmission,kb_fuel,kb_drivetrain,brand
fold 3,vehicle_age,ENGINE,gb_mileage,MODEL_SERIES (text),DAMAGE,segment,kb_body_type,kb_transmission,kb_fuel,brand,kb_drivetrain
fold 4,vehicle_age,ENGINE,gb_mileage,MODEL_SERIES (text),DAMAGE,segment,kb_body_type,kb_transmission,kb_fuel,kb_drivetrain,brand
fold 5,vehicle_age,ENGINE,gb_mileage,MODEL_SERIES (text),DAMAGE,segment,kb_body_type,kb_transmission,kb_fuel,kb_drivetrain,brand


... (4 outputs left out)

**motor gücü bantları**

,alt hp,üst hp,ortalama SHAP
0,0,150,-0.161318
1,150,200,0.074605
2,200,250,0.144123
3,250,1000000000,0.358959


**yaş bantları**

,alt yaş,üst yaş,ortalama SHAP
0,0,5,0.576997
1,5,9,0.332656
2,9,13,0.084332
3,13,17,0.270933
4,17,100,0.610956


![tr-sh-01-importance](../reports/figures/tr-sh-01-importance.png)

![tr-sh-02-beeswarm](../reports/figures/tr-sh-02-beeswarm.png)

![tr-sh-03-dependence](../reports/figures/tr-sh-03-dependence.png)

![tr-sh-04-cohorts](../reports/figures/tr-sh-04-cohorts.png)

![tr-sh-05-km-age](../reports/figures/tr-sh-05-km-age.png)

### shap/04_variants
[betik](../analysis/shap/04_variants.py) · [kart](../analysis/cards/shap/04_variants.json)

In [8]:
Markdown(card("shap/04_variants"))

**Soru:** Servis edilen üç modeli (LightGBM, CatBoost SVD, CatBoost native) hangi öznitelikler sürüklüyor?

**Veri:** load_clean() (29.988 ilan) ve data/serving'deki final modeller.

**Yöntem:** Her modelde bütün veride SHAP (örnekleme yok), toplanabilirlik kontrolü, gruplu ve ayrı önem tabloları, beeswarm figürleri. Native CatBoost atfı koşumdan koşuma iki durum arasında gidip geliyor (muaf, belgelenmiş).

**Bölme:** `none`

**Sızıntı:**
- aynı ilan iki tarafta: `n/a`
- hedef öznitelikte: `none` — SHAP, fiyattan türeyen öznitelik içermeyen modelin tahminlerini açıklıyor.
- bütün veriye uydurma: `present` — Tasarım gereği: servis edilen, bütün veride kurulmuş final modeller açıklanıyor; performans iddiası yok.
- zamansal: `n/a`

In [9]:
m = recorded("shap/04_variants", docs)

**Kayıttan okunuyor** (ağır betik: servis modelleriyle bütün veride SHAP).

In [10]:
sf = m["shap_final"]
show("yöntem ve toplamsallık", flat({"domain.shap": m["domain"]["shap"], "add_err": sf["add_err"], "base_log": sf["base_log"]}))
for key in ("final_model_table", "final_model_table_separate", "catboost_tfidf_svd", "catboost_tfidf_svd_separate",
            "catboost_native", "catboost_native_separate", "catboost_native_combined"):
    show(key, pd.DataFrame([[lb(g, "tr"), g, x, p] for g, x, p in sf[key]], columns=["grup", "kimlik", "ortalama |SHAP|", "pay %"]))

**yöntem ve toplamsallık**

,değer
domain.shap.n_used,29988
domain.shap.method,"TÜM VERİ (örnekleme YOK). Grup SHAP = satır bazında üye SHAP toplamı (işaretli net), sonra ortalama-mutlak. LGB→shap..."
domain.shap.note,"SVD boyutları (model_0..) → MODEL_SERIES gruplandı. ENGINE (hp+cc), DAMAGE (boya/değişen/hasar) gruplu. Native ham t..."
domain.shap.png,"shap_plots/beeswarm_catboost_native.png, shap_plots/beeswarm_catboost_svd.png, shap_plots/beeswarm_lightgbm.png, sha..."
add_err,0.0
base_log,14.299832


**final_model_table**

,grup,kimlik,ortalama |SHAP|,pay %
0,Yaş (yıl),vehicle_age,0.2871,43.9
1,Motor (hp + cc),ENGINE,0.1363,20.8
2,Kilometre,gb_mileage,0.0846,12.9
3,Model/seri adı,MODEL_SERIES (text),0.0577,8.8
4,Hasar (panel + ağır hasar),DAMAGE,0.0422,6.5
5,Segment,segment,0.0341,5.2
6,Kasa tipi,kb_body_type,0.0066,1.0
7,Vites,kb_transmission,0.0044,0.7
8,Yakıt,kb_fuel,0.0008,0.1
9,Çekiş,kb_drivetrain,0.0002,0.0


**final_model_table_separate**

,grup,kimlik,ortalama |SHAP|,pay %
0,Yaş (yıl),vehicle_age,0.2871,37.1
1,Model/seri adı,MODEL_SERIES (text),0.1493,19.3
2,Motor (hp + cc),ENGINE,0.1392,18.0
3,Kilometre,gb_mileage,0.0846,10.9
4,Hasar (panel + ağır hasar),DAMAGE,0.0683,8.8
5,Segment,segment,0.0341,4.4
6,Kasa tipi,kb_body_type,0.0066,0.8
7,Vites,kb_transmission,0.0044,0.6
8,Yakıt,kb_fuel,0.0008,0.1
9,Çekiş,kb_drivetrain,0.0002,0.0


... (6 outputs left out)

**catboost_native_separate**

,grup,kimlik,ortalama |SHAP|,pay %
0,Yaş (yıl),vehicle_age,0.2112,30.5
1,Kilometre,gb_mileage,0.1468,21.2
2,Motor (hp + cc),ENGINE,0.1168,16.8
3,model,model,0.1090,15.7
4,Hasar (panel + ağır hasar),DAMAGE,0.0586,8.5
5,Kasa tipi,kb_body_type,0.0189,2.7
6,series,series,0.0100,1.4
7,Vites,kb_transmission,0.0090,1.3
8,Marka,brand,0.0075,1.1
9,Çekiş,kb_drivetrain,0.0031,0.4


**catboost_native_combined**

,grup,kimlik,ortalama |SHAP|,pay %
0,Yaş (yıl),vehicle_age,0.2112,32.1
1,Kilometre,gb_mileage,0.1468,22.3
2,Motor (hp + cc),ENGINE,0.1127,17.1
3,Model/seri adı,MODEL_SERIES (text),0.1099,16.7
4,Hasar (panel + ağır hasar),DAMAGE,0.0363,5.5
5,Kasa tipi,kb_body_type,0.0189,2.9
6,Vites,kb_transmission,0.0090,1.4
7,Marka,brand,0.0075,1.1
8,Çekiş,kb_drivetrain,0.0031,0.5
9,Yakıt,kb_fuel,0.0022,0.3


### shap/06_one_listing
[betik](../analysis/shap/06_one_listing.py) · [kart](../analysis/cards/shap/06_one_listing.json)

In [11]:
Markdown(card("shap/06_one_listing"))

**Soru:** Tek bir ilanın tahmini, OOF SHAP'la adım adım nasıl açıklanır?

**Veri:** load_clean() ve 02_oof_shap'ın SHAP dizileri (run_id kapısı).

**Yöntem:** Temsili bir ilan seçilir; taban değerden tahmine giden şelale, en büyük katkılar görünür, kalanlar katlanır; iki dilli figür.

**Bölme:** `oof_input`

**Sızıntı:**
- aynı ilan iki tarafta: `n/a`
- hedef öznitelikte: `none` — SHAP, fiyattan türeyen öznitelik içermeyen modelin tahminlerini açıklıyor.
- bütün veriye uydurma: `n/a`
- zamansal: `n/a`

In [12]:
script = run("shap/06_one_listing")           # [1]–[5]: yükle + hesapla; kaydetmez
m = script.to_metrics(script.res)
check(m, docs["shap/06_one_listing"])

118i Standart 2007 · [('vehicle_age', -0.613), ('MODEL_SERIES (text)', -0.14), ('power_hp_val', -0.115)]


**Kayıtla aynı:** defterde hesaplanan sonuç `metrics/` kaydıyla birebir aynı.

In [13]:
c = m["shap_case"]
show("tek ilan", flat({k: v for k, v in c.items() if k not in ("top", "km_label")}))
show("en büyük katkılar", pd.DataFrame([[lb(g, "tr"), x, lbl["tr"]] for g, x, lbl in c["top"]], columns=["grup", "SHAP", "değer"]))

**tek ilan**

,değer
name,118i Standart
year,2007
price,595000.0
pred,519679.769841
rest,-0.268145
n_rest,21
visible,"vehicle_age, MODEL_SERIES (text), power_hp_val, is_heavy_damaged, gb_mileage, hood_state, segment, door_painted, kb_..."
n_other,15
wf_max,10
n_items,24


**en büyük katkılar**

,grup,SHAP,değer
0,Yaş (yıl),-0.612511,19 yıl
1,Model/seri adı,-0.140429,118i Standart
2,Motor gücü (hp),-0.115397,138 hp


![tr-sh-06-waterfall](../reports/figures/tr-sh-06-waterfall.png)